# Experiment Notebook Template

Use this template to create a new experiment notebook (e.g., `m_00X.ipynb`).

This structure follows project requirements: section content note first, then code, with clear functions for each stage.


## 1) Metadata and Imports
**Content:** Define ModelID, runtime metadata, paths, and imports used across the notebook.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

MODEL_ID = "m_00X"
RUNNER = "codex"
MACHINE = "cpu"
DATA_SCOPE = "full"

BASE_DIR = Path.cwd().resolve().parents[0]  # model_training
assert BASE_DIR.name == "model_training", f"Unexpected BASE_DIR: {BASE_DIR}"


## 2) Data Loading
**Content:** Load train/validation factor and target files from canonical locations.


In [ ]:
def load_data(base_dir: Path):
    X_train = pd.read_csv(base_dir / "training_data" / "train_df_factor.csv")
    y_train = pd.read_csv(base_dir / "training_data" / "train_df_target.csv")
    X_val = pd.read_csv(base_dir / "val_data" / "val_df_factor.csv")
    y_val = pd.read_csv(base_dir / "val_data" / "val_df_target.csv")
    return X_train, y_train, X_val, y_val

X_train, y_train, X_val, y_val = load_data(BASE_DIR)


## 3) Required Data Checks
**Content:** Validate shapes, alignment, schema consistency, missingness, and target distribution.


In [ ]:
def run_required_checks(X_train, y_train, X_val, y_val):
    checks = {
        "train_shape": X_train.shape,
        "target_shape": y_train.shape,
        "val_shape": X_val.shape,
        "val_target_shape": y_val.shape,
        "schema_match": list(X_train.columns) == list(X_val.columns),
        "duplicate_columns": int(X_train.columns.duplicated().sum()),
        "missing_train": int(X_train.isna().sum().sum()),
        "missing_val": int(X_val.isna().sum().sum()),
        "target_mean": float(y_train.iloc[:, 0].mean()),
        "val_positive_rate": float(y_val.iloc[:, 0].mean()),
    }
    return checks

checks = run_required_checks(X_train, y_train, X_val, y_val)
checks


## 4) Preprocessing and Features
**Content:** Build preprocessing pipeline for numeric/categorical columns without data leakage.


In [ ]:
def build_preprocessor(X: pd.DataFrame):
    num_cols = X.select_dtypes(include=["number"]).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median"))])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    return preprocessor, num_cols, cat_cols

preprocessor, num_cols, cat_cols = build_preprocessor(X_train)


## 5) Model Training
**Content:** Define model and fit pipeline using training set only.


In [ ]:
from sklearn.linear_model import LogisticRegression

def train_model(X_train, y_train, preprocessor):
    model = LogisticRegression(max_iter=1000, class_weight="balanced", solver="liblinear")
    pipe = Pipeline([("preprocessor", preprocessor), ("model", model)])
    pipe.fit(X_train, y_train.iloc[:, 0])
    return pipe

pipeline = train_model(X_train, y_train, preprocessor)


## 6) Validation Scoring (Official)
**Content:** Predict on validation set and compute score with `help_stuff/validation_score.py` official logic.


In [ ]:
import sys
sys.path.append(str(BASE_DIR / "help_stuff"))
from validation_score import score as official_score

def evaluate_model(pipeline, X_val, y_val, threshold=0.5):
    proba = pipeline.predict_proba(X_val)[:, 1]
    pred = (proba >= threshold).astype(int)
    val_score = official_score(y_val.iloc[:, 0].values, pred)
    return {"score": float(val_score), "threshold": threshold}

result = evaluate_model(pipeline, X_val, y_val, threshold=0.5)
result


## 7) Optional Artifact Saving
**Content:** Save artifacts only when required by policy (default off for predictions/models in PRs).


In [ ]:
def maybe_save_artifacts(save_artifacts=False):
    if not save_artifacts:
        print("Artifact saving skipped (default).")
        return
    # Example placeholder paths:
    # BASE_DIR / "train_nb" / f"{MODEL_ID}_val_pred.csv"
    # BASE_DIR / "trained_models" / f"{MODEL_ID}_model.pkl"
    print("Implement artifact saving only when explicitly requested.")

maybe_save_artifacts(save_artifacts=False)


## 8) Training Log Entry Template
**Content:** Prepare a markdown-ready entry to append to `training_log.md` (append-only; never overwrite).


In [ ]:
def build_log_stub(model_id, checks, result):
    return f"""### ModelID: {model_id} (executed)

Date:

YYYY-MM-DD

Runner:

{RUNNER}

Machine:

{MACHINE}

Data Scope:

{DATA_SCOPE}

Validation Score:

{result['score']}

Notes:

* Train shape: {checks['train_shape']}
* Validation shape: {checks['val_shape']}
* Schema match: {checks['schema_match']}
"""

print(build_log_stub(MODEL_ID, checks, result))
